## Импорты

In [ ]:
import json
import logging
import os
import pickle
import re
import time
import warnings
from datetime import datetime
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import backoff
import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image
from tqdm.auto import tqdm

import clip
from sentence_transformers import SentenceTransformer, CrossEncoder

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_qdrant import FastEmbedSparse, RetrievalMode
from langchain_qdrant.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.http.models import Distance

from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, HTML

import gdown
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
log = logging.getLogger(__name__)
load_dotenv(find_dotenv())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Тестируемые документы

In [2]:
TEST_QUERIES = [
    "My sister just had a baby and I want to give her something really useful. What should I get?",
    "I'm adopting a rescue dog this weekend. What should I prepare at home before bringing him?",
    "I want to start cooking healthy meals instead of ordering takeout. Where do I even begin?",
    "We just moved into a new apartment and everything is empty. What are the first things to buy?",
    "I'm starting a YouTube channel but I have zero equipment. What do I absolutely need on day one?",
    "I have two weeks off and want to completely disconnect from screens. How can I keep myself entertained?",
    "I have an important job interview tomorrow and I want to look my absolute best. What can help?",
    "My 7-year-old is bored during summer break and I want to keep him learning. Any ideas?",
    "Our office is organizing a weekend team-building outdoors. What should we bring for group activities?",
    "I want to pick up a creative hobby that I can practice quietly in my apartment at night. What do I need?",
    "I'm about to drive cross-country alone for three days straight. How do I survive the boredom and stay safe?",
    "I'm leaving my house empty for a month while traveling. How do I make sure everything stays safe?",
    "I want to make personalized Christmas gifts for my family this year instead of buying them. What do I need?",
    "My doctor told me I need to lose 15 kg and reduce stress. What products can support this lifestyle change?",
    "I got accepted to a semester abroad in a cold country and I've never experienced winter. What should I pack?",
]

## Config

In [ ]:
class Config:
    def __init__(self):
        self.device = DEVICE
        self.max_products = 20000
        self.use_all_data = True
        self.force_rebuild = False

        self.data_dir = "check_data"
        self.models_dir = "models"
        self.results_dir = "search_results"

        self.clip_model = "ViT-B/32"
        self.st_model = "sentence-transformers/clip-ViT-B-32-multilingual-v1"
        self.rerank_model = "cross-encoder/ms-marco-MiniLM-L-6-v2"
        self.sparse_model = "Qdrant/bm25"

        self.qdrant_host = "localhost"
        self.qdrant_port = 6333
        self.qdrant_url = f"http://{self.qdrant_host}:{self.qdrant_port}"

        self.collection_baseline = "products_baseline"
        self.collection_text = "products_text"
        self.collection_multimodal = "products_multimodal"
        self.collection_taxonomy = "taxonomy_index"
        self.collection_taxonomy_products = "products_taxonomy"

        self.llm_api_key = '...'
        self.llm_base_url = "https://api.vsellm.ru/v1"
        self.llm_model = "openai/gpt-4.1-mini"
        self.use_llm = True

        self.search_limit = 5
        self.taxonomy_limit = 3
        self.use_reranking = False
        self.fusion_alpha = 0.6

        self.taxonomy_validation = True

        for d in [self.data_dir, self.models_dir, self.results_dir]:
            Path(d).mkdir(parents=True, exist_ok=True)

    @property
    def image_embeddings_path(self):
        return f"{self.data_dir}/image_embeddings_all.pkl"

In [23]:
def _cache_load(path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

def _cache_save(path, obj):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    log.info(f"Cached: {path}")

## Подгрузка датасета

In [4]:
def load_dataset(cfg):
    path = "all_products_clean.parquet"
    if not os.path.exists(path):
        gdown.download("https://drive.google.com/uc?id=1cDIVIuZjxqEVuqWz_ue0M1CiXrF9UlPx", path, quiet=False)
    df = pd.read_parquet(path).dropna(subset=["name"])
    if not cfg.use_all_data:
        df = df.iloc[: cfg.max_products]
    log.info(f"Dataset: {len(df)} products")
    return df

## LLM prompt для подзапросов

In [5]:
SYSTEM_PROMPT_SUBQUERY = """You are an experienced e-commerce expert and product strategist. \
Your task is to analyze the user's query and decompose it into specific, actionable subqueries \
that can be used to search for relevant products.

Identify the key activities, needs, and contextual requirements mentioned in the query. \
Break them down into distinct, semantically clear subqueries that capture all aspects of the user's requirements.

For example, given the query:
"I'm going to the beach, but along the way I want to visit the mountains. What should I bring?"

You should generate subqueries such as:
1. Beach essentials for swimming and sun exposure
2. Mountain gear for hiking and camping
3. Travel accessories for car, bus, or airplane journey
"""

USER_PROMPT_SUBQUERY = """For the following user query: "{query}", generate a list of specific and meaningful \
subqueries that represent the different product categories or needs implied by the scenario.

Return your response in strict JSON format:
{{
  "subquery_1": "description_1",
  "subquery_2": "description_2",
  "subquery_3": "description_3"
}}
Subqueries must NOT be questions. They should be concrete e-commerce product descriptions \
containing enough information to semantically search for the desired product.
Do not create more than 4 subqueries!
"""

@backoff.on_exception(backoff.expo, Exception, max_tries=5, max_time=300)
def _llm_create_subqueries(query, model, cfg):
    client = OpenAI(api_key=cfg.llm_api_key, base_url=cfg.llm_base_url)
    resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_SUBQUERY},
                {"role": "user", "content": USER_PROMPT_SUBQUERY.format(query=query)},
            ],
            temperature=0, max_tokens=200,
        )
    raw = resp.choices[0].message.content.replace("\n", "").replace("json", "").replace("`", "")
    return json.loads(raw)

def generate_subqueries(cfg, query):
    result = _llm_create_subqueries(query, cfg.llm_model, cfg)
    subs = list(result.values()) if isinstance(result, dict) else []
    log.info(f"Подзапросов ({len(subs)}): {subs}")
    return [query] + subs

## Эмбеддинги

In [6]:
class STEmbeddings(Embeddings):
    def __init__(self, cfg):
        self.model = SentenceTransformer(cfg.st_model).to(cfg.device)
    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()
    def embed_query(self, text):
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

class FusedEmbeddings(Embeddings):
    def __init__(self, cfg, fused_map=None):
        self.model = SentenceTransformer(cfg.st_model).to(cfg.device)
        self._map = fused_map or {}
    def embed_documents(self, texts):
        return [
            self._map[t] if t in self._map
            else self.model.encode([t], normalize_embeddings=True)[0].tolist()
            for t in texts
        ]
    def embed_query(self, text):
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

class QueryFuser:
    def __init__(self, cfg):
        self.model = SentenceTransformer(cfg.st_model).to(cfg.device)
    def fuse(self, query, subqueries):
        all_texts = [query] + subqueries
        embs = self.model.encode(all_texts, normalize_embeddings=True)
        avg = np.mean(embs, axis=0)
        return (avg / np.linalg.norm(avg)).tolist()

## Image Loader

In [7]:
class ImageLoader:
    def __init__(self, pool_size=25):
        self.session = requests.Session()
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=pool_size, pool_maxsize=pool_size
        )
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)
        self._failed = set()
        self.ok = self.fail = 0
    def load(self, url, timeout=5):
        if not url or url in self._failed:
            self.fail += 1
            return None
        try:
            r = self.session.get(url, timeout=timeout)
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            self.ok += 1
            return img
        except Exception:
            self._failed.add(url)
            self.fail += 1
            return None
    def stats(self):
        t = self.ok + self.fail
        log.info(f"Изображений: {self.ok}/{t} загружено ({self.ok / max(t, 1) * 100:.0f}%)")

class VisualEncoder:
    def __init__(self, cfg):
        self.device = cfg.device
        self.model, self.preprocess = clip.load(cfg.clip_model, device=cfg.device)
        self.model.eval()
    def encode(self, images, bs=32):
        out = []
        for i in tqdm(range(0, len(images), bs), desc="Encoding images"):
            batch = torch.stack([self.preprocess(im) for im in images[i:i+bs]]).to(self.device)
            with torch.no_grad():
                e = self.model.encode_image(batch).cpu().numpy()
            out.append(e / np.linalg.norm(e, axis=1, keepdims=True))
        return np.vstack(out)

In [8]:
def precompute_image_embeddings_parallel(df, cfg, force=False, workers=20, chunk_size=5000):
    cached = None if force else _cache_load(cfg.image_embeddings_path)
    if cached is not None:
        log.info(f"Загружены кэшированные эмбеддинги изображений: {cached['embeddings'].shape}")
        return cached["embeddings"], cached["valid_indices"]
    vis = VisualEncoder(cfg)
    loader = ImageLoader(pool_size=workers + 5)

    all_embs, all_idx = [], []
    rows = [(i, str(row.get("image", "")) if pd.notna(row.get("image")) else "")
            for i, (_, row) in enumerate(df.iterrows())]

    for chunk_start in tqdm(range(0, len(rows), chunk_size), desc="Chunks"):
        chunk = rows[chunk_start:chunk_start + chunk_size]
        images_map = {}

        def _load(args):
            idx, url = args
            return idx, loader.load(url) if url else None

        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = [ex.submit(_load, item) for item in chunk]
            for f in as_completed(futures):
                idx, img = f.result()
                if img is not None:
                    images_map[idx] = img

        if not images_map:
            continue

        valid_idx = sorted(images_map.keys())
        images = [images_map[i] for i in valid_idx]
        embs = vis.encode(images)
        all_embs.append(embs)
        all_idx.extend(valid_idx)

        del images, images_map, embs
        import gc; gc.collect()
        log.info(f"чанк обработан, всего на данный момент: {len(all_idx)} изображений")

    loader.stats()
    if not all_embs:
        return np.array([]), []
    
    final_embs = np.vstack(all_embs)
    log.info(f"Создано {len(final_embs)} эмбеддингов изображений")
    _cache_save(cfg.image_embeddings_path, {"embeddings": final_embs, "valid_indices": all_idx})
    return final_embs, all_idx

In [9]:
def compute_fused_embeddings(docs, cfg, image_embs, valid_indices):
    idx_to_img = {}
    for i, idx in enumerate(valid_indices):
        v = image_embs[i]
        idx_to_img[idx] = v / np.linalg.norm(v)

    text_model = SentenceTransformer(cfg.st_model).to(cfg.device)
    texts = [d.page_content for d in docs]
    text_embs = text_model.encode(texts, normalize_embeddings=True)

    alpha = cfg.fusion_alpha
    fused, n_fused = [], 0
    for i, doc in enumerate(docs):
        df_idx = doc.metadata.get("df_index")
        if df_idx is not None and df_idx in idx_to_img:
            c = alpha * text_embs[i] + (1 - alpha) * idx_to_img[df_idx]
            fused.append((c / np.linalg.norm(c)).tolist())
            n_fused += 1
        else:
            fused.append(text_embs[i].tolist())
            
    log.info(f"Объединено: {n_fused}/{len(docs)} документов с изображениями (alpha={alpha})")
    return fused

## Validation Taxonomy

In [10]:
TAXONOMY_VALIDATE_SYSTEM = """You are an e-commerce taxonomy expert. \
Given a user query and a list of candidate product categories, \
determine which categories are genuinely relevant to the query intent."""

TAXONOMY_VALIDATE_USER = """Query: "{query}"
Candidate categories:
{categories}

For each category, decide if it's relevant (1) or not (0).
If none are relevant, suggest up to 3 better category search keywords.

Return ONLY valid JSON (no markdown):
{{
  "validations": {{"category_name": 0 or 1, ...}},
  "all_irrelevant": true/false,
  "suggested_keywords": ["keyword1", "keyword2"]
}}"""

In [11]:
class TaxonomyValidator:
    def __init__(self, cfg):
        self.cfg = cfg
        self.client = OpenAI(api_key=cfg.llm_api_key, base_url=cfg.llm_base_url)

    def validate(self, query, taxonomies):
        if not taxonomies:
            return [], True, []
        cats = "\n".join(f"- {t}" for t in taxonomies)

        resp = self.client.chat.completions.create(
                model=self.cfg.llm_model,
                messages=[
                    {"role": "system", "content": TAXONOMY_VALIDATE_SYSTEM},
                    {"role": "user", "content": TAXONOMY_VALIDATE_USER.format(
                        query=query, categories=cats
                    )},
                ],
                temperature=0, max_tokens=250,
            )
        raw = re.sub(r"", "", resp.choices[0].message.content.strip())
        data = json.loads(raw)
        validations = data.get("validations", {})
        good = [t for t in taxonomies if validations.get(t, 0) == 1]
        all_bad = data.get("all_irrelevant", len(good) == 0)
        suggestions = data.get("suggested_keywords", [])
        return good, all_bad, suggestions

## Гибридный поиск

In [12]:
class HybridVectorStore:
    def __init__(self, cfg, collection_name, dense_embeddings=None):
        self.cfg = cfg
        self.name = collection_name
        self.client = QdrantClient(url=cfg.qdrant_url)
        self.dense = dense_embeddings or STEmbeddings(cfg)
        self.sparse = FastEmbedSparse(model_name=cfg.sparse_model)
        self._store = None

    def build(self, docs, force=False):
        if self.client.collection_exists(self.name) and not force:
            log.info(f"Коллекция '{self.name}' существует, пропускаем создание")
            return
        if self.client.collection_exists(self.name):
            self.client.delete_collection(self.name)
            log.info(f"Удалена старая коллекция '{self.name}'")
        self._store = QdrantVectorStore.from_documents(
            docs, embedding=self.dense, sparse_embedding=self.sparse,
            url=self.cfg.qdrant_url, collection_name=self.name,
            retrieval_mode=RetrievalMode.HYBRID, distance=Distance.COSINE,
        )
        log.info(f"Коллекция '{self.name}': {len(docs)} документов")

    def _get_store(self):
        if self._store is None:
            self._store = QdrantVectorStore(
                client=self.client, embedding=self.dense, sparse_embedding=self.sparse,
                collection_name=self.name, retrieval_mode=RetrievalMode.HYBRID,
                distance=Distance.COSINE,
            )
        return self._store

    def search(self, query, k=5):
        results = self._get_store().similarity_search_with_score(query, k=k)
        return [{"content": d.page_content, "metadata": d.metadata, "score": s} for d, s in results]

    def search_by_vector(self, vector, k=5):
        results = self.client.query_points(
            collection_name=self.name, query=vector,
            limit=k, with_payload=True,
        )
        return [
            {"content": p.payload.get("page_content", ""),
             "metadata": p.payload.get("metadata", {}), "score": p.score}
            for p in results.points
        ]

    def search_by_vector_with_filter(self, vector, metadata_key, values, k=5):
        results = self.client.query_points(
            collection_name=self.name, query=vector,
            limit=k, with_payload=True,
            query_filter=models.Filter(must=[
                models.FieldCondition(
                    key=f"metadata.{metadata_key}",
                    match=models.MatchAny(any=values),
                )
            ]),
        )
        return [
            {"content": p.payload.get("page_content", ""),
             "metadata": p.payload.get("metadata", {}), "score": p.score}
            for p in results.points
        ]

## ReRanker - опционально 

In [13]:
class Reranker:
    def __init__(self, cfg):
        self.active = cfg.use_reranking
        if self.active:
            try:
                self.model = CrossEncoder(cfg.rerank_model)
            except Exception:
                self.active = False

    def __call__(self, query, results):
        if not self.active or not results:
            return results
        pairs = [[query, f"{r['metadata'].get('name', '')} {r['content']}"[:512]] for r in results]
        scores = self.model.predict(pairs)
        for r, s in zip(results, scores):
            r["rerank_score"] = float(s)
        return sorted(results, key=lambda x: x["rerank_score"], reverse=True)

## document builder

In [14]:
def build_baseline_docs(df):
    docs = []
    for _, row in df.iterrows():
        name = str(row.get("name", ""))
        if not name.strip():
            continue
        docs.append(Document(
            page_content=name,
            metadata={"name": name[:200]},
        ))
    return docs


def build_text_docs(df):
    docs = []
    for _, row in df.iterrows():
        name = str(row.get("name", ""))
        desc = str(row.get("description", ""))[:500] if pd.notna(row.get("description")) else ""
        content = f"{name} {desc}".strip()
        if not content:
            continue
        taxonomy = str(row.get("taxonomy", "")) if pd.notna(row.get("taxonomy")) else ""
        docs.append(Document(
            page_content=content,
            metadata={"name": name[:200], "description": desc[:300], "taxonomy": taxonomy},
        ))
    return docs


def build_multimodal_docs(df, image_embs, valid_indices):
    valid_set = set(valid_indices)
    docs = []
    for i, (_, row) in enumerate(df.iterrows()):
        name = str(row.get("name", ""))
        desc = str(row.get("description", ""))[:500] if pd.notna(row.get("description")) else ""
        content = f"{name} {desc}".strip()
        if not content:
            continue
        taxonomy = str(row.get("taxonomy", "")) if pd.notna(row.get("taxonomy")) else ""
        docs.append(Document(
            page_content=content,
            metadata={"name": name[:200], "description": desc[:300], "taxonomy": taxonomy,
                       "has_image": i in valid_set, "df_index": i,
                       "image_url": str(row.get("image", "")) if pd.notna(row.get("image")) else ""},
        ))
    return docs


def build_taxonomy_docs(df):
    taxonomies = df["taxonomy"].dropna().unique().tolist()
    return [Document(page_content=t, metadata={"taxonomy": t}) for t in taxonomies if t.strip()]

## Поисковики

In [15]:
class Baseline:
    def __init__(self, cfg, store, reranker):
        self.cfg, self.store, self.reranker = cfg, store, reranker

    def search(self, query):
        t0 = time.time()
        results = self.store.search(query, k=self.cfg.search_limit)
        results = self.reranker(query, results)
        return {"method": "baseline", "query": query, "subqueries": [query],
                "results": results, "search_time": time.time() - t0}

class TextSearcher:
    def __init__(self, cfg, store, reranker):
        self.cfg, self.store, self.reranker = cfg, store, reranker
        self.fuser = QueryFuser(cfg)

    def search(self, query):
        t0 = time.time()
        subqueries = generate_subqueries(self.cfg, query)
        fused_vector = self.fuser.fuse(query, subqueries[1:])
        results = self.store.search_by_vector(fused_vector, k=self.cfg.search_limit)
        results = self.reranker(query, results)
        return {"method": "text_searcher", "query": query, "subqueries": subqueries,
                "results": results, "search_time": time.time() - t0}

class MultimodalSearcher:
    def __init__(self, cfg, store, reranker):
        self.cfg, self.store, self.reranker = cfg, store, reranker
        self.fuser = QueryFuser(cfg)

    def search(self, query):
        t0 = time.time()
        subqueries = generate_subqueries(self.cfg, query)
        fused_vector = self.fuser.fuse(query, subqueries[1:])
        results = self.store.search_by_vector(fused_vector, k=self.cfg.search_limit)
        results = self.reranker(query, results)
        return {"method": "multimodal", "query": query, "subqueries": subqueries,
                "results": results, "search_time": time.time() - t0}

class TaxonomySearcher:
    def __init__(self, cfg, taxonomy_store, product_store, reranker):
        self.cfg = cfg
        self.taxonomy_store = taxonomy_store
        self.product_store = product_store
        self.reranker = reranker
        self.fuser = QueryFuser(cfg)
        self.validator = TaxonomyValidator(cfg) if (cfg.use_llm and cfg.taxonomy_validation) else None

    def _find_taxonomies(self, query, k=None):
        k = k or self.cfg.taxonomy_limit
        results = self.taxonomy_store.search(query, k=k)
        return [r["metadata"]["taxonomy"] for r in results if r["metadata"].get("taxonomy")]

    def _validate_and_fix(self, query, taxonomies):
        if not self.validator:
            return taxonomies, False
        good, all_bad, suggestions = self.validator.validate(query, taxonomies)
        if good:
            dropped = set(taxonomies) - set(good)
            if dropped:
                log.info(f"Категория проверена: оставлено {good}, отброшено {dropped}")
            return good, False
        if all_bad and suggestions:
            log.info(f"Все категории нерелевантны, повтор с ключевыми словами: {suggestions}")
            retry = []
            for kw in suggestions[:3]:
                retry.extend(self._find_taxonomies(kw, k=2))
            retry = list(dict.fromkeys(retry))[:self.cfg.taxonomy_limit]
            if retry:
                good2, still_bad, _ = self.validator.validate(query, retry)
                if good2:
                    log.info(f"Повторная попытка с переопределеинем категории успешна: {good2}")
                    return good2, False
        log.warning(f"Хороших категорий не найдено, возврат к исходным: {taxonomies}")
        return taxonomies, True

    def search(self, query):
        t0 = time.time()

        taxonomies = self._find_taxonomies(query)
        log.info(f"Начальные taxonomies: {taxonomies}")

        if not taxonomies:
            return {"method": "taxonomy_searcher", "query": query, "subqueries": [query],
                    "taxonomies_found": [], "taxonomy_fallback": True,
                    "results": [], "search_time": time.time() - t0}

        taxonomies, fallback = self._validate_and_fix(query, taxonomies)

        subqueries = generate_subqueries(self.cfg, query)
        fused_vector = self.fuser.fuse(query, subqueries[1:])
        results = self.product_store.search_by_vector_with_filter(
            vector=fused_vector, metadata_key="taxonomy",
            values=taxonomies, k=self.cfg.search_limit,
        )
        results = self.reranker(query, results)
        return {
            "method": "taxonomy_searcher", "query": query, "subqueries": subqueries,
            "taxonomies_found": taxonomies, "taxonomy_fallback": fallback,
            "results": results, "search_time": time.time() - t0,
        }

## LLM AS A JUDGE

In [16]:
JUDGE_SYSTEM_PROMPT = """You are an expert e-commerce search quality evaluator. \
You assess how well a product matches a user's search query.

Scoring guidelines:

## on_topic (0 or 1)
- 1: the product is primarily about or strongly relevant to the query intent
- 0: the product is NOT primarily about the query, even if keywords partially match

## relevance (0-10)
- 0: completely unrelated product
- 1-2: same broad department but wrong category
- 3-4: related category but poor fit
- 5-6: reasonable match, partially satisfies the need
- 7-8: good match, directly addresses the query
- 9-10: perfect match with ideal specifications

## probability_to_click (0.0-1.0)
- 0.0-0.2: user would scroll past without noticing
- 0.3-0.5: user might glance but unlikely to click
- 0.6-0.8: user would likely click to learn more
- 0.9-1.0: user would almost certainly click immediately

## probability_to_buy (0.0-1.0)
- 0.0-0.1: would never buy, wrong product entirely
- 0.1-0.3: very unlikely, significant mismatch
- 0.3-0.5: possible but user would prefer better
- 0.5-0.7: decent option, user might buy if no better results
- 0.7-0.9: strong match, user would likely purchase
- 0.9-1.0: exactly what the user is looking for

Important:
- on_topic should be 1 ONLY if the product directly addresses the user's need
- probability_to_buy should ALWAYS be ≤ probability_to_click
- relevance 0-4 → probability_to_buy < 0.3, on_topic should be 0
- relevance 7+ → probability_to_buy > 0.5, on_topic should be 1
- Be strict: generic or loosely related products should score low"""

JUDGE_USER_PROMPT = (
    'Query: "{query}"\n'
    'Product: {name}\n'
    'Description: {desc}\n\n'
    'Return ONLY valid JSON (no markdown, no explanation):\n'
    '{{"on_topic": <int 0 or 1>, '
    '"relevance": <int 0-10>, '
    '"probability_to_click": <float 0.0-1.0>, '
    '"probability_to_buy": <float 0.0-1.0>}}'
)

In [17]:
class Evaluator:
    def __init__(self, cfg):
        self.cfg = cfg

    def judge(self, query, results):
        evals = []
        client = OpenAI(api_key=self.cfg.llm_api_key, base_url=self.cfg.llm_base_url)
        for i, r in enumerate(results[:self.cfg.search_limit]):
            name = r["metadata"].get("name", "")
            desc = r["content"][:300]
            try:
                resp = client.chat.completions.create(
                    model=self.cfg.llm_model,
                    messages=[
                        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                        {"role": "user", "content": JUDGE_USER_PROMPT.format(
                            query=query, name=name, desc=desc
                        )},
                    ],
                    temperature=0.1, max_tokens=120,
                )
                raw = re.sub(r"\n\s*|\s*\n", "", resp.choices[0].message.content.strip())
                data = json.loads(raw)
                on_topic = 1 if int(data.get("on_topic", 0)) == 1 else 0
                relevance = min(10, max(0, int(data.get("relevance", 0))))
                p_click = min(1.0, max(0.0, float(data.get("probability_to_click", 0))))
                p_buy = min(1.0, max(0.0, float(data.get("probability_to_buy", 0))))
                p_buy = min(p_buy, p_click)
            except Exception as e:
                log.warning(f"LLM judge error: {e}")
                on_topic, relevance, p_click, p_buy = 0, 0, 0.0, 0.0
            evals.append({
                "position": i + 1, "product_name": name,
                "on_topic": on_topic, "relevance": relevance,
                "probability_to_click": p_click, "probability_to_buy": p_buy,
                "hybrid_score": r.get("score", 0), "rerank_score": r.get("rerank_score"),
            })
        return evals

    def metrics(self, search_result, evaluations):
        if not evaluations:
            return {"method": search_result["method"], "query": search_result["query"]}
        rels = [e["relevance"] for e in evaluations]
        clicks = [e["probability_to_click"] for e in evaluations]
        buys = [e["probability_to_buy"] for e in evaluations]
        on_topics = [e["on_topic"] for e in evaluations]
        k = len(evaluations)
        dcg = sum(rels[i] / np.log2(i + 2) for i in range(len(rels)))
        idcg = sum(sorted(rels, reverse=True)[i] / np.log2(i + 2) for i in range(len(rels)))
        otr_at_k = sum(on_topics) / k if k > 0 else 0.0
        return {
            "method": search_result["method"],
            "query": search_result["query"][:80],
            "search_time_s": round(search_result.get("search_time", 0), 4),
            "avg_relevance": round(float(np.mean(rels)), 2),
            "avg_p_click": round(float(np.mean(clicks)), 3),
            "avg_p_buy": round(float(np.mean(buys)), 3),
            "ndcg": round(float(dcg / idcg) if idcg > 0 else 0.0, 4),
            "p_at_1_relevant": 1 if rels and rels[0] >= 7 else 0,
            "otr_at_k": round(otr_at_k, 3),
            "taxonomy_fallback": search_result.get("taxonomy_fallback", False),
            "evaluations": evaluations,
        }

def _img_tag(url, width=80):
    if url and url.strip():
        return (f'<img src="{url}" width="{width}" '
                f'style="border-radius:4px;" onerror="this.style.display=\'none\'">')
    return '<span style="color:#ccc;">no img</span>'

def _has_images(method):
    return method in ("multimodal", "taxonomy_searcher")

def print_query_results(query_idx, query, method, results, evaluations=None, taxonomy_fallback=False):
    eval_map = {e["position"]: e for e in evaluations} if evaluations else {}
    show_img = _has_images(method)

    html = f'<h4>Запрос {query_idx}: {query}</h4>'
    html += f'<p><b>Метод:</b> {method.upper()}'
    if taxonomy_fallback:
        html += ' <span style="color:#e67e22; font-weight:bold;">⚠ taxonomy fallback</span>'
    html += '</p>'

    if not results:
        html += '<p style="color:red;">нет результатов</p>'
        display(HTML(html))
        return

    html += '<table style="border-collapse:collapse; width:100%;">'
    header = '<tr style="background:#f0f0f0;"><th style="padding:6px;">Поз</th>'
    if show_img:
        header += '<th style="padding:6px;">Фото</th>'
    header += ('<th style="padding:6px; text-align:left;">Товар</th>'
               '<th style="padding:6px;">Score</th>'
               '<th style="padding:6px;">OT</th>'
               '<th style="padding:6px;">Rel</th>'
               '<th style="padding:6px;">P(click)</th>'
               '<th style="padding:6px;">P(buy)</th></tr>')
    html += header

    for i, r in enumerate(results):
        pos = i + 1
        name = r["metadata"].get("name", "")[:80]
        url = r["metadata"].get("image_url", "")
        score = r.get("rerank_score", r.get("score", 0))
        ev = eval_map.get(pos, {})

        on_topic = ev.get("on_topic", "—")
        rel = ev.get("relevance", "—")
        p_click = ev.get("probability_to_click", "—")
        p_buy = ev.get("probability_to_buy", "—")

        ot_color = "#2ecc71" if on_topic == 1 else "#e74c3c" if on_topic == 0 else "#999"
        ot_text = "✓" if on_topic == 1 else "✗" if on_topic == 0 else "—"
        rel_color = ("#2ecc71" if isinstance(rel, int) and rel >= 7 else
                     "#f39c12" if isinstance(rel, int) and rel >= 4 else
                     "#e74c3c" if isinstance(rel, int) else "#999")
        p_click_str = f"{p_click:.2f}" if isinstance(p_click, float) else "—"
        p_buy_str = f"{p_buy:.2f}" if isinstance(p_buy, float) else "—"

        html += f'<tr style="border-bottom:1px solid #eee;"><td style="padding:6px; text-align:center;">{pos}</td>'
        if show_img:
            html += f'<td style="padding:6px; text-align:center;">{_img_tag(url)}</td>'
        html += (f'<td style="padding:6px;">{name}</td>'
                 f'<td style="padding:6px; text-align:center;">{score:.3f}</td>'
                 f'<td style="padding:6px; text-align:center; color:{ot_color}; font-weight:bold;">{ot_text}</td>'
                 f'<td style="padding:6px; text-align:center; color:{rel_color}; font-weight:bold;">{rel}</td>'
                 f'<td style="padding:6px; text-align:center;">{p_click_str}</td>'
                 f'<td style="padding:6px; text-align:center;">{p_buy_str}</td></tr>')

    html += '</table>'
    if evaluations:
        rels = [e["relevance"] for e in evaluations]
        ots = [e["on_topic"] for e in evaluations]
        otr = sum(ots) / len(ots) if ots else 0
        html += (f'<p style="margin-top:4px; color:#555;">'
                 f'OTR@{len(ots)}: {otr:.0%} | '
                 f'Avg rel: {np.mean(rels):.1f} | Min: {min(rels)} | Max: {max(rels)}</p>')

    display(HTML(html))

def print_all_results(all_search_results, all_evaluations=None):
    queries_seen = []
    for method, query_map in all_search_results.items():
        for q in query_map:
            if q not in queries_seen:
                queries_seen.append(q)
    for qi, query in enumerate(queries_seen, 1):
        for method in all_search_results:
            res_list = all_search_results[method].get(query, [])
            for sr in res_list:
                evals = None
                if all_evaluations and method in all_evaluations:
                    evals = all_evaluations[method].get(query)
                fallback = sr.get("taxonomy_fallback", False)
                print_query_results(qi, query, method, sr["results"], evals, fallback)

## Сохранение 

In [18]:
def save_result(result, cfg):
    show_img = _has_images(result["method"])
    out = {
        "timestamp": datetime.now().isoformat(),
        "method": result["method"], "query": result["query"],
        "subqueries": result.get("subqueries", []),
        "taxonomies_found": result.get("taxonomies_found", []),
        "taxonomy_fallback": result.get("taxonomy_fallback", False),
        "search_time": result.get("search_time", 0),
        "results": [
            {"score": r.get("rerank_score", r.get("score", 0)),
             "name": r["metadata"].get("name", ""),
             **({"image_url": r["metadata"].get("image_url", "")} if show_img else {}),
             "content": r["content"][:300]}
            for r in result["results"][:10]
        ],
    }
    path = f"{cfg.results_dir}/{result['method']}_{datetime.now():%Y%m%d_%H%M%S}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

## Pipeline

In [ ]:
class SearchPipeline:
    def __init__(self, cfg):
        self.cfg = cfg
        self.df = None
        self.reranker = None
        self.searchers = {}
        self.image_embs = None
        self.image_valid_idx = None

    def load_data(self):
        self.df = load_dataset(self.cfg)
        return self

    def precompute_images(self, force=False):
        self.image_embs, self.image_valid_idx = precompute_image_embeddings_parallel(self.df, self.cfg, force)
        return self

    def _reranker(self):
        if self.reranker is None:
            self.reranker = Reranker(self.cfg)
        return self.reranker

    def _build_fused_store(self, collection_name, force=False):
        docs = build_multimodal_docs(self.df, self.image_embs, self.image_valid_idx)
        fused = compute_fused_embeddings(docs, self.cfg, self.image_embs, self.image_valid_idx)
        fused_map = {doc.page_content: vec for doc, vec in zip(docs, fused)}
        dense = FusedEmbeddings(self.cfg, fused_map)
        store = HybridVectorStore(self.cfg, collection_name, dense_embeddings=dense)
        store.build(docs, force=force)
        return store

    def build_baseline(self, force=False):
        log.info("Создание Baseline")
        docs = build_baseline_docs(self.df)
        store = HybridVectorStore(self.cfg, self.cfg.collection_baseline)
        store.build(docs, force=force)
        self.searchers["baseline"] = Baseline(self.cfg, store, self._reranker())
        return self

    def build_text_searcher(self, force=False):
        log.info("Создание TextSearcher")
        docs = build_text_docs(self.df)
        store = HybridVectorStore(self.cfg, self.cfg.collection_text)
        store.build(docs, force=force)
        self.searchers["text_searcher"] = TextSearcher(self.cfg, store, self._reranker())
        return self

    def build_multimodal(self, force=False):
        log.info("Создание MultimodalSearcher (text+image fused)")
        store = self._build_fused_store(self.cfg.collection_multimodal, force=force)
        self.searchers["multimodal"] = MultimodalSearcher(self.cfg, store, self._reranker())
        return self

    def build_taxonomy_searcher(self, force=False):
        log.info("Создание TaxonomySearcher (таксономия + текст + изображения объединены)")
        tax_docs = build_taxonomy_docs(self.df)
        tax_store = HybridVectorStore(self.cfg, self.cfg.collection_taxonomy)
        tax_store.build(tax_docs, force=force)
        prod_store = self._build_fused_store(self.cfg.collection_taxonomy_products, force=force)
        self.searchers["taxonomy_searcher"] = TaxonomySearcher(
            self.cfg, tax_store, prod_store, self._reranker(),
        )
        return self

    def run(self, queries=None, methods=None):
        queries = queries or TEST_QUERIES
        methods = methods or list(self.searchers.keys())
        all_results = {m: {} for m in methods if m in self.searchers}
        for qi, query in enumerate(queries, 1):
            log.info(f"Запрос {qi}/{len(queries)}: {query}")
            for m in methods:
                if m not in self.searchers:
                    continue
                r = self.searchers[m].search(query)
                all_results[m].setdefault(query, []).append(r)
                save_result(r, self.cfg)
                fallback = r.get("taxonomy_fallback", False)
                print_query_results(qi, query, m, r["results"], taxonomy_fallback=fallback)
                extra = "[FALLBACK]" if fallback else ""
                log.info(f"{m}: {r['search_time']:.3f}s, {len(r['results'])} results{extra}")
        return all_results

    def evaluate(self, all_results):
        ev = Evaluator(self.cfg)
        all_evaluations = {}
        summary_rows = []
        for method, query_map in all_results.items():
            all_evaluations[method] = {}
            log.info(f"\nEvaluating: {method.upper()}")
            for query, res_list in query_map.items():
                for sr in res_list:
                    evals = ev.judge(sr["query"], sr["results"])
                    m = ev.metrics(sr, evals)
                    all_evaluations[method][query] = evals
                    summary_rows.append({
                        "Method": method, "Query": query[:60],
                        "Time(s)": m.get("search_time_s", 0),
                        "OTR@K": m.get("otr_at_k", 0),
                        "Avg Rel": m.get("avg_relevance", 0),
                        "P(click)": m.get("avg_p_click", 0),
                        "P(buy)": m.get("avg_p_buy", 0),
                        "NDCG": m.get("ndcg", 0),
                        "P@1": m.get("p_at_1_relevant", 0),
                        "Tax.Fallback": m.get("taxonomy_fallback", False),
                    })
        print_all_results(all_results, all_evaluations)
        df_summary = pd.DataFrame(summary_rows)
        if not df_summary.empty:
            print(df_summary.to_string(index=False))
            agg_cols = {
                "Time(s)": "mean", "OTR@K": "mean", "Avg Rel": "mean",
                "P(click)": "mean", "P(buy)": "mean", "NDCG": "mean", "P@1": "mean",
            }
            df_avg = df_summary.groupby("Method").agg(agg_cols).round(3).reset_index()
            if "Tax.Fallback" in df_summary.columns:
                fb = df_summary.groupby("Method")["Tax.Fallback"].mean().round(3).reset_index()
                fb.columns = ["Method", "Fallback%"]
                df_avg = df_avg.merge(fb, on="Method", how="left").fillna(0)
            print("\n Средние по методам:")
            print(df_avg.to_string(index=False))
            path = f"{self.cfg.results_dir}/evaluation_{datetime.now():%Y%m%d_%H%M%S}.json"
            with open(path, "w", encoding="utf-8") as f:
                json.dump({"summary": summary_rows, "evaluations": {
                    m: {q: evs for q, evs in qmap.items()} for m, qmap in all_evaluations.items()
                }}, f, indent=2, ensure_ascii=False, default=str)
        return all_evaluations, df_summary

## Запуск

In [ ]:
cfg = Config()
log.info(f"device={cfg.device}, taxonomy_validation={cfg.taxonomy_validation}")

pipe = SearchPipeline(cfg)
pipe.load_data()
pipe.precompute_images(force=True)

pipe.build_baseline(force=True)
pipe.build_text_searcher(force=True)
pipe.build_multimodal(force=True)
pipe.build_taxonomy_searcher(force=True)

all_results = pipe.run(TEST_QUERIES)
all_evals, df_summary = pipe.evaluate(all_results)

df_summary.to_csv(f"{cfg.results_dir}/evaluation_summary.csv", index=False)
df_avg = df_summary.groupby("Method").agg({
    "Time(s)": "mean", "OTR@K": "mean", "Avg Rel": "mean",
    "P(click)": "mean", "P(buy)": "mean", "NDCG": "mean", "P@1": "mean",
}).round(3).reset_index()
df_avg.to_csv(f"{cfg.results_dir}/evaluation_avg_by_method.csv", index=False)
print(df_avg.to_string(index=False))